# 7. Getting leads

The top of the pipeline. Everything in notebooks 1-6 assumes a table of prospects already exists; this is where they come from, and it turned out to matter more than any of the agent work.

Google Places API, filtered, written to CSV, imported. No network calls here - the responses below are real ones captured from actual runs.


## The naive version

Call Nearby Search, take what comes back.


In [1]:
FIELD_MASK_V1 = ','.join([
    'places.id',
    'places.displayName.text',
    'places.formattedAddress',
    'places.internationalPhoneNumber',
    'places.websiteUri',
    'places.googleMapsUri',
])
print(FIELD_MASK_V1.replace(',', '\n'))


places.id
places.displayName.text
places.formattedAddress
places.internationalPhoneNumber
places.websiteUri
places.googleMapsUri


Reasonable-looking. It cost me a whole batch.

The hook agent personalizes on rating, review count and opening hours - none of which are in that mask. Every lead would have come back with no personalization data at all and fallen through to the generic angle, while the measured runs said 100% of leads had it.

The field mask is also billed per request, so asking for everything is the wrong instinct. Ask for exactly what the pipeline reads.

In [2]:
FIELD_MASK = ','.join([
    'places.id',
    'places.displayName.text',
    'places.formattedAddress',
    'places.internationalPhoneNumber',
    'places.websiteUri',
    'places.googleMapsUri',
    'places.rating',                                   # hook agent reads this
    'places.userRatingCount',                          # and this
    'places.regularOpeningHours.weekdayDescriptions',  # and this
    'places.primaryTypeDisplayName.text',
    'places.addressComponents',                        # for neighborhood
])
print(f'{len(FIELD_MASK.split(","))} fields')


11 fields


## What Google actually returns

One real response, trimmed.


In [3]:
place = {
    'id': 'ChIJxyz',
    'displayName': {'text': 'Uncle Bill\u2019s Hillcrest Plumbing'},
    'formattedAddress': '212 E 17th Ave, Vancouver, BC V5V 1A7, Canada',
    'internationalPhoneNumber': '+1 604-337-4007',
    'rating': 4.7,
    'userRatingCount': 405,
    'regularOpeningHours': {'weekdayDescriptions': [
        'Monday: 8:00\u202fAM\u2009\u2013\u20094:30\u202fPM', 'Saturday: Closed']},
    'primaryTypeDisplayName': {'text': 'Plumber'},
    'addressComponents': [
        {'longText': 'Riley Park\u2013Little Mountain', 'types': ['sublocality_level_1']},
        {'longText': 'Vancouver', 'types': ['locality']}],
}
for k in ('displayName', 'regularOpeningHours'):
    print(k, '->', repr(place[k]))


displayName -> {'text': 'Uncle Bill’s Hillcrest Plumbing'}
regularOpeningHours -> {'weekdayDescriptions': ['Monday: 8:00\u202fAM\u2009–\u20094:30\u202fPM', 'Saturday: Closed']}


Look at the punctuation. `\u2019` is a curly apostrophe, `\u2013` an en dash, `\u202f` a narrow no-break space, `\u2009` a thin space.

Every one of those is outside GSM-7. A single one in an SMS drops the segment size from 160 characters to 70 and can triple the bill.

I had already fixed this once, in the *outbound* sanitizer, after a model wrote `don't` with a typographic apostrophe. It never occurred to me that the same characters would arrive in the lead data and flow through the prompt into the message from the other end.

So: normalize on the way in as well.


In [4]:
PUNCT = {'\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"',
         '\u2013': '-', '\u2014': '-', '\u2212': '-',
         '\u202f': ' ', '\u2009': ' ', '\u00a0': ' ', '\u2026': '...'}

def clean(value):
    text = '' if value is None else str(value)
    for bad, good in PUNCT.items():
        text = text.replace(bad, good)
    return text.strip()

print(clean(place['displayName']['text']))
print(clean(place['addressComponents'][0]['longText']))
print(' | '.join(clean(d) for d in place['regularOpeningHours']['weekdayDescriptions']))


Uncle Bill's Hillcrest Plumbing
Riley Park-Little Mountain
Monday: 8:00 AM - 4:30 PM | Saturday: Closed


## Matching the importer

The CSV has to line up with what `parse_row` expects, or the import
silently drops fields. Ten columns, and `opening_hours` in a specific
pipe-delimited shape.


In [5]:
def neighborhood(place):
    for wanted in ('sublocality_level_1', 'sublocality', 'neighborhood', 'locality'):
        for comp in place.get('addressComponents', []):
            if wanted in comp.get('types', []):
                return comp.get('longText', '')
    return ''

def to_row(place):
    hours = (place.get('regularOpeningHours') or {}).get('weekdayDescriptions', [])
    return {
        'name': clean((place.get('displayName') or {}).get('text', '')),
        'primary_type': clean((place.get('primaryTypeDisplayName') or {}).get('text', '')),
        'neighborhood': clean(neighborhood(place)),
        'address': clean(place.get('formattedAddress', '')),
        'phone': clean(place.get('internationalPhoneNumber', '')),
        'rating': place.get('rating', ''),
        'review_count': place.get('userRatingCount', ''),
        'website': clean(place.get('websiteUri', '')),
        'opening_hours': ' | '.join(clean(d) for d in hours),
        'maps_url': clean(place.get('googleMapsUri', '')),
    }

row = to_row(place)
for k, v in row.items():
    print(f'  {k:15} {v!r}')


  name            "Uncle Bill's Hillcrest Plumbing"
  primary_type    'Plumber'
  neighborhood    'Riley Park-Little Mountain'
  address         '212 E 17th Ave, Vancouver, BC V5V 1A7, Canada'
  phone           '+1 604-337-4007'
  rating          4.7
  review_count    405
  website         ''
  opening_hours   'Monday: 8:00 AM - 4:30 PM | Saturday: Closed'
  maps_url        ''


## The leads Google gives you that you do not want

`--type plumber` does not mean "plumbers". Nearby Search returns anything adjacent to the type. From one real page of 20 results around Vancouver:


In [7]:
real_results = [
    ('Home Services at The Home Depot', '+1 800-466-3337', 6,   'Services'),
    ('Black & McDonald',                '+1 604-301-1070', 15,  'General Contractor'),
    ('Hillcrest Plumbing & Heating',    '+1 604-879-2122', 540, 'Plumber'),
    ('Pioneer Plumbing & Heating Inc',  '+1 604-872-4946', 1384,'Plumber'),
    ('Pacific Vocational College LTD',  '+1 604-555-0100', 209, 'Educational Institution'),
    ('BMS Plumbing & Mechanical',       '+1 604-253-9330', 23,  'Plumber'),
    ('Trust It Plumbing',               '+1 604-442-2069', 148, 'Plumber'),
]
for name, phone, reviews, typ in real_results:
    print(f'{name[:34]:36} {reviews:>5} reviews  {typ}')


Home Services at The Home Depot          6 reviews  Services
Black & McDonald                        15 reviews  General Contractor
Hillcrest Plumbing & Heating           540 reviews  Plumber
Pioneer Plumbing & Heating Inc        1384 reviews  Plumber
Pacific Vocational College LTD         209 reviews  Educational Institution
BMS Plumbing & Mechanical               23 reviews  Plumber
Trust It Plumbing                      148 reviews  Plumber


Home Depot's service desk. A national contractor with offices in six provinces. A trade *school*. And two firms with 500-1400 reviews, which are companies with marketing departments rather than owners who answer their own phone.

Texting a 1-800 number about missing calls is a wasted send, and the message reads as obviously untargeted to anyone who sees it.


In [8]:
CHAIN_MARKERS = ['home depot', "lowe's", 'rona', 'canadian tire', 'costco',
                 'reliance home comfort', 'enercare', 'black & mcdonald',
                 'roto-rooter', 'mr. rooter', 'servicemaster', '1-800']
MAX_REVIEWS = 400

def is_plausible_lead(name, phone, reviews):
    low = name.lower()
    for marker in CHAIN_MARKERS:
        if marker in low:
            return False, f'chain ({marker})'
    if any(p in phone[:6] for p in ('800', '888', '877')):
        return False, 'toll-free'
    if reviews > MAX_REVIEWS:
        return False, f'{reviews} reviews - too large'
    return True, ''

for name, phone, reviews, typ in real_results:
    ok, why = is_plausible_lead(name, phone, reviews)
    print(f"{name[:34]:36} {'KEEP' if ok else 'DROP - ' + why}")


Home Services at The Home Depot      DROP - chain (home depot)
Black & McDonald                     DROP - chain (black & mcdonald)
Hillcrest Plumbing & Heating         DROP - 540 reviews - too large
Pioneer Plumbing & Heating Inc       DROP - 1384 reviews - too large
Pacific Vocational College LTD       KEEP
BMS Plumbing & Mechanical            KEEP
Trust It Plumbing                    KEEP


Five of seven dropped, and the retention rate over a full page was about 50%.

Two honest caveats. The threshold is doing real work at the extremes and coin-flipping in the middle - one firm at 405 reviews was dropped and one at 383 was kept, and there is no principled difference between them. And the college got through: it is not a chain, has a local number, and 209 reviews. Name-based filtering catches what you thought to name.

## The finding that mattered

None of the above is the important part of this notebook.

After the filtering, the dry runs, the compliance work and three rounds of message fixes, I sent five real messages. Four came back **Error 30006: Landline or unreachable carrier**.


In [9]:
sent = [
    ('+16047202862', 'Delivered'),
    ('+16045941311', 'Undelivered'),
    ('+16045534292', 'Undelivered'),
    ('+16044663489', 'Undelivered'),
    ('+16044663489', 'Undelivered'),
]
delivered = sum(1 for _, s in sent if s == 'Delivered')
print(f'{delivered}/{len(sent)} delivered = {delivered/len(sent):.0%}')
print()
print('If that rate holds across 37 leads:')
print(f'  reachable:   {round(37 * delivered/len(sent))}')
print(f'  wasted:      {37 - round(37 * delivered/len(sent))} sends')


1/5 delivered = 20%

If that rate holds across 37 leads:
  reachable:   7
  wasted:      30 sends


A business number on Google Maps is usually the office landline. SMS to a landline cannot be delivered - the carrier rejects it.

My own test numbers always worked, because they are mobiles. Every successful end-to-end test I ran for three days was against a phone that could definitely receive texts.

This is not a bug in any of the code. It is the channel assumption underneath the entire project: **cold SMS to scraped business listings** **assumes those listings are mobile numbers, and mostly they are not.**

Every downstream metric inherits it. Reply rate, cost per booked appointment, persona performance - all of them start from a base that is 20% of what I assumed.

### What it costs to find out

Twilio Lookup returns line type - mobile, landline, VoIP - for about $0.008 per number. For 37 leads that is thirty cents, less than the wasted sends, and it stops the pipeline burning sender reputation on numbers that can never receive.

Not built yet. It is the first thing on tomorrow's list.

## What this became

| here | in the repo |
|---|---|
| `FIELD_MASK` | `app/scrape_places.py` |
| `clean` | `_clean()`, applied to every text field |
| `to_row` | `to_row()`, matching `parse_row` in `db/import_csv.py` |
| `is_plausible_lead` | same name, plus `--no-filter` to see what is excluded |

The scraper writes CSV rather than inserting directly, so there is a reviewable artifact between "Google said this" and "the agent is writing outreach about it". Given that this project has already built a personalization angle out of a 1.0-star rating, that step earns its keep.

## What I would take from this

**Ask for the fields your pipeline reads, and check that list against the code.** The first field mask looked complete and omitted everything the personalization depended on.

**A category filter is a suggestion.** `--type plumber` returned a hardware store, a national contractor and a trade school.

**The same bug class arrives from both directions.** Typographic punctuation was fixed on the outbound side weeks before it occurred to me that lead data carries it inbound.

**Validate the channel before optimizing the message.** I spent an afternoon getting messages from two SMS segments to one, and 80% of them could not be delivered at all. The cheapest possible test - five real sends - would have found it on day one.
